# Demo 6: Network Impairments on HD Video

**Course**: Future Media Internet
**Duration**: ~10 min
**Environment**: Kaggle Notebook (CPU, NumPy + Matplotlib + SciPy)

## Objective
Simulate packet loss, jitter, and bandwidth drops on a synthetic HD test frame.
Show how each impairment type creates visually distinct artifacts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
print('Imports OK')


In [ ]:
# Create synthetic HD test frame (scaled to 540x960)
H, W = 540, 960

def create_test_frame():
    frame = np.zeros((H, W, 3), dtype=np.float32)
    colors = [(1,0,0),(0,1,0),(0,0,1),(1,1,0),(0,1,1),(1,0,1),(1,1,1)]
    bar_w = W // len(colors)
    for i, c in enumerate(colors):
        frame[:H//3, i*bar_w:(i+1)*bar_w] = c
    for x in range(W):
        frame[H//3:2*H//3, x] = (x/W, 0.5, 1-x/W)
    for y in range(2*H//3, H, 20):
        for x in range(0, W, 4):
            v = 0.9 if (x//4+y//20)%2==0 else 0.2
            frame[y:y+10, x:x+2] = v
    return frame

original = create_test_frame()
fig, ax = plt.subplots(figsize=(12,7))
ax.imshow(np.clip(original,0,1))
ax.set_title('Original HD Test Frame (540x960)')
ax.axis('off')
plt.show()


In [ ]:
# Network impairment simulators

def packet_loss(frame, rate):
    result = frame.copy()
    bh, bw = 30, 40
    nb_h, nb_w = H//bh, W//bw
    mask = np.random.random((nb_h, nb_w)) < rate
    for i in range(nb_h):
        for j in range(nb_w):
            if mask[i,j]:
                result[i*bh:(i+1)*bh, j*bw:(j+1)*bw] = 0.5
    return result

def jitter(frame, px):
    result = frame.copy()
    shifts = (np.random.randn(H)*px).astype(int)
    for y in range(H):
        s = shifts[y]
        result[y] = np.roll(result[y], s, axis=0)
    return result

def bw_drop(frame, scale):
    h2, w2 = int(H*scale), int(W*scale)
    low = ndimage.zoom(frame, (scale,scale,1), order=1)
    up = ndimage.zoom(low, (1/scale,1/scale,1), order=1)
    return np.clip(up[:H,:W], 0, 1)

print('Simulators ready')


In [ ]:
# Full comparison grid
np.random.seed(42)
loss_rates = [0.01, 0.05, 0.10, 0.20]
jitter_px = [1, 3, 6, 10]
bw_scales = [0.75, 0.50, 0.25, 0.10]

fig, axes = plt.subplots(3, 5, figsize=(18, 12))

for row, (label, levels, func) in enumerate([
    ('Packet Loss', loss_rates, lambda f,l: packet_loss(f,l)),
    ('Jitter', jitter_px, lambda f,l: jitter(f,l)),
    ('BW Drop', bw_scales, lambda f,l: bw_drop(f,l)),
]):
    axes[row,0].imshow(original)
    axes[row,0].set_title('Original', fontweight='bold')
    for j, lv in enumerate(levels):
        imp = func(original, lv)
        axes[row,j+1].imshow(np.clip(imp,0,1))
        axes[row,j+1].set_title('{} {}'.format(label, lv))
for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Network Impairments on HD Video', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Detail zoom comparison
dy, dx = slice(200,350), slice(400,550)
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

axes[0,0].imshow(original[dy,dx])
axes[0,0].set_title('Original Detail', fontweight='bold')
axes[0,0].axis('off')

pl = packet_loss(original, 0.10)
axes[0,1].imshow(np.clip(pl[dy,dx],0,1))
axes[0,1].set_title('Packet Loss 10%')
axes[0,1].axis('off')

jt = jitter(original, 6)
axes[0,2].imshow(np.clip(jt[dy,dx],0,1))
axes[0,2].set_title('Jitter 6px')
axes[0,2].axis('off')

bw = bw_drop(original, 0.25)
axes[1,0].imshow(np.clip(bw[dy,dx],0,1))
axes[1,0].set_title('BW Drop 25%')
axes[1,0].axis('off')

cb = packet_loss(bw_drop(original, 0.5), 0.05)
axes[1,1].imshow(np.clip(cb[dy,dx],0,1))
axes[1,1].set_title('BW 50% + Loss 5%')
axes[1,1].axis('off')

cb2 = jitter(packet_loss(original, 0.05), 3)
axes[1,2].imshow(np.clip(cb2[dy,dx],0,1))
axes[1,2].set_title('Loss 5% + Jitter 3px')
axes[1,2].axis('off')

plt.suptitle('Detail Zoom: Artifact Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Key Observations

- **Packet loss**: missing blocks (gray squares). Error concealment helps but leaves visible artifacts.
- **Jitter**: horizontal tearing (lines misaligned). Critical for live streaming where no buffering is possible.
- **Bandwidth drop**: overall blur from forced resolution downgrade.
- **Combined**: compound artifacts are worse than any single impairment.

## Real-world mitigation
- YouTube/Netflix: ABR (Adaptive Bitrate) + FEC (Forward Error Correction) + jitter buffers
- WebRTC/Google Meet: NACK (retransmission) + FEC + adaptive encoding
- 5G URLLC: ultra-low latency reduces jitter sensitivity
